# M10 — protótipo de treino (Colab Pro)

**O objetivo deste notebook não é um modelo bom. É um número:** o **custo por época**.

O plano de treino (`docs/plans/m10-treino-vastai.md`) estima a Fase 3 em $600–1.500 por
proporção ao run de M5 — cujo custo real **não está registrado em lugar nenhum do repositório**.
Esse é o risco R1, e é o número mais frágil de todo o plano. Um run de ~300 h aqui mede o custo
por época de verdade e permite reprecificar antes de comprometer centenas de dólares.

## O que este notebook NÃO faz

- não treina até convergir (são poucas épocas, de propósito)
- não decide a arquitetura — isso é o bake-off de T4
- não produz artefato publicável

## Configuração alvo

| | |
|---|---|
| GPU | **L4** (melhor custo/hora normalizado no Colab Pro) |
| Corpus | ~300 h do TAGARELA (~31 shards) |
| Features | fbank 80-dim em **lilcom_chunky** — 33 MB/h `[MEDIDO]` |
| Disco | ~10 GB de features + ~17 GB de parquets temporários |
| Checkpoints | no Drive, para sobreviver à sessão |


## 1. Ambiente

Colab Pro **não tem background execution** (isso é Pro+). A aba precisa ficar aberta.
Com checkpoint por época, uma desconexão vira atraso, não perda.


In [ ]:
import subprocess, sys, os, json, time
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| disponivel', torch.cuda.is_available())
print('disco livre:', subprocess.run(['df','-h','/content'], capture_output=True, text=True).stdout.splitlines()[-1])


## 2. Dependências

⚠️ **O ponto mais frágil do notebook é o `k2`**: a wheel precisa casar exatamente com a versão
de torch e CUDA do runtime. Se a célula falhar, confira a matriz em
<https://k2-fsa.github.io/k2/installation/pre-compiled-cuda-wheels-linux/> e ajuste a URL.

O `k2stub` do repositório **não serve aqui** — ele cobre só a ativação Swoosh para smoke em
CPU, e a receita de treino usa `k2.ctc_loss` e grafos.


In [ ]:
# --- k2: escolher a wheel EXATA, nunca deixar o pip resolver ---
# O PyPI tem um pacote chamado "k2" que NAO e este: ele instala um __init__.py
# sem a extensao compilada _k2, e o erro so aparece no import.
# Por isso aqui nao existe "pip install k2": a wheel e escolhida pelo indice
# oficial e instalada por URL exata.
import re, urllib.request, subprocess, sys

INDEX   = 'https://k2-fsa.github.io/k2/cuda.html'
PY_TAG  = f'cp{sys.version_info.major}{sys.version_info.minor}'
TORCH   = torch.__version__.split('+')[0]
CUDA    = torch.version.cuda or ''
print(f'runtime: python {PY_TAG}  torch {TORCH}  cuda {CUDA}')

html  = urllib.request.urlopen(INDEX, timeout=120).read().decode('utf-8', 'replace')
urls  = sorted(set(re.findall(r'https://[^"\s]+\.whl', html)))

def campos(u):
    m = re.search(r'k2-([\d.]+(?:\.dev\d+)?)\+cuda([\d.]+)\.torch([\d.]+)-(cp\d+)-', u)
    return m.groups() if m else None   # (versao, cuda, torch, pytag)

cand = [(c, u) for u in urls if (c := campos(u))
        and c[3] == PY_TAG and c[2] == TORCH]
if not cand:
    raise RuntimeError(
        f'Nenhuma wheel k2 para {PY_TAG} + torch {TORCH}.\n'
        f'Torch disponiveis para {PY_TAG}: '
        + ', '.join(sorted({c[2] for u in urls if (c := campos(u)) and c[3] == PY_TAG}))
        + '\nOpcao: fixar o torch do runtime numa versao coberta.')

exato = [x for x in cand if x[0][1] == CUDA]
if exato:
    escolha = exato
else:
    # Subir de CUDA minor e a direcao perigosa: uma wheel de 12.9 pode exigir
    # driver mais novo do que o runtime de 12.4 oferece. Preferir o maior minor
    # que seja <= o do runtime, e so subir se nao houver nenhum abaixo.
    def minor(v):
        p = v.split('.')
        return (int(p[0]), int(p[1]) if len(p) > 1 else 0)
    alvo   = minor(CUDA)
    mesmos = [x for x in cand if minor(x[0][1])[0] == alvo[0]]
    if not mesmos:
        raise RuntimeError(f'k2 existe para torch {TORCH}/{PY_TAG}, mas nao para CUDA {alvo[0]}.x. '
                           f'CUDAs: {sorted({c[1] for c, _ in cand})}')
    abaixo  = [x for x in mesmos if minor(x[0][1]) <= alvo]
    escolha = abaixo or mesmos
    usado   = max(minor(x[0][1]) for x in escolha)
    escolha = [x for x in escolha if minor(x[0][1]) == usado]
    print(f'AVISO: sem wheel para CUDA {CUDA} exata; usando {usado[0]}.{usado[1]}'
          + ('' if abaixo else ' (ACIMA do runtime -- pode exigir driver mais novo)')
          + '. Compatibilidade de minor version costuma funcionar, mas NAO esta verificada aqui.')

(ver, cu, tv, _), url = sorted(escolha, key=lambda x: x[0][0])[-1]
print(f'instalando k2 {ver} (cuda {cu}, torch {tv}) -- ~177 MB')

# --no-deps: a wheel declara torch como dependencia e o pip trocaria o torch do
# runtime, o que quebra a CUDA da sessao.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', url], check=True)
# lilcom NAO vem com o lhotse: `from lhotse.features.io import LilcomChunkyWriter`
# importa sem erro e a falha so aparece quando um worker abre o storage, depois de
# todo o filtro de corpus ter rodado. Ver requirements-train.txt.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lhotse', 'lilcom', 'kaldialign', 'sentencepiece', 'kaldi_native_fbank',
                'num2words', 'regex'], check=True)
import lilcom  # falha aqui, nao dentro de um worker 20 min depois
print('lilcom', lilcom.__version__ if hasattr(lilcom, '__version__') else 'ok')

import importlib.metadata as md
import k2

# A versao vem do dist-info, nao de um atributo do modulo: k2 nem sempre expoe
# __version__, e a string do dist-info carrega a tag local (+cuda12.8.torch2.11.0),
# que e justamente a prova de QUAL wheel ficou instalada.
instalado = md.version('k2')
print('k2', instalado)
assert f'cuda{cu}' in instalado and f'torch{tv}' in instalado, (
    f'wheel instalada ({instalado}) nao e a escolhida (cuda {cu}, torch {tv}) -- '
    'provavelmente sobrou o pacote errado do PyPI; Runtime > Restart session e rode de novo')

# Prova funcional: um kernel do k2 rodando na GPU. E um teste, nao uma suposicao
# sobre o nome de um atributo -- se a API mudou, o notebook mostra o que existe
# em vez de morrer em AttributeError.
try:
    _r = k2.RaggedTensor([[1, 2], [3]]).to(torch.device('cuda'))
    print('k2 com CUDA: ok (RaggedTensor em', _r.device, ')')
except Exception as e:
    print('AVISO: prova funcional de CUDA nao rodou:', type(e).__name__, e)
    print('atributos publicos de k2:', [a for a in dir(k2) if not a.startswith('_')][:40])
    print('A versao instalada esta correta; siga, mas o primeiro passo do treino'
          ' e quem vai confirmar que os kernels funcionam.')


## 3. Código: icefall + jvscribe

O `jvscribe` traz os preparadores de corpus já testados (`prep_tagarela.py`), com filtro
determinístico de alucinação e relatório de cobertura por show.


In [ ]:
%cd /content

# Ambos publicos: clone direto, sem token.
!test -d icefall  || git clone -q https://github.com/k2-fsa/icefall.git
# --branch workspace NAO e detalhe: a default do repo e main, que esta 125 commits
# atras e AINDA GRAVA FEATURES EM NUMPY (115 MB/h contra 33 MB/h do lilcom).
!test -d jvscribe || git clone -q --branch workspace https://github.com/paulohenriquevn/jvscribe.git

PREP = '/content/jvscribe/jvscribe/finetune/prep_tagarela.py'
assert os.path.isdir('/content/icefall'), 'clone do icefall falhou'
assert os.path.exists(PREP), (
    'clone do jvscribe falhou -- na versao anterior desta celula isso passava em '
    'silencio e so aparecia tres celulas depois')

# Confere o CONTEUDO, nao so a presenca: um clone da branch errada tem o arquivo
# e grava numpy. Falhar aqui custa um segundo; descobrir depois do fbank custa
# a extracao inteira.
assert 'LilcomChunkyWriter' in open(PREP).read(), (
    'prep_tagarela.py sem LilcomChunkyWriter -- branch errada. '
    'Use --branch workspace.')

os.environ['PYTHONPATH'] = '/content/icefall:' + os.environ.get('PYTHONPATH', '')
!pip install -q -r /content/icefall/requirements.txt 2>&1 | tail -3
print('icefall + jvscribe (workspace) prontos')


## 4. Checkpoints no Drive

A sessão morre; o Drive não. **Só os checkpoints vão para o Drive** — as features ficam no
disco local, que é ordens de magnitude mais rápido para o dataloader.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EXP = '/content/drive/MyDrive/jvscribe/m10_proto/exp'
os.makedirs(EXP, exist_ok=True)
print('checkpoints ->', EXP)


## 5. Corpus

Amostragem **estratificada**: shards espaçados uniformemente ao longo dos 1.764. Como
**um shard é um show** (`wiki/medicoes/m10-t3-quanto-corpus-pt-existe.md`), espaçar por
posição é espaçar por show.

Cada shard tem ~9,62 h `[MEDIDO]`, então 31 shards ≈ 300 h.


In [ ]:
N_SHARDS = 31           # ~300 h; cada shard ~9,62 h [MEDIDO]
TOTAL_SHARDS = 1764
RAW = '/content/tagarela_raw'; os.makedirs(RAW, exist_ok=True)
from huggingface_hub import hf_hub_download
idx = [round(i*(TOTAL_SHARDS-1)/(N_SHARDS-1)) for i in range(N_SHARDS)]
t0 = time.time()
for k, i in enumerate(idx):
    f = f'data/train-{i:05d}-of-01764.parquet'
    hf_hub_download('freds0/TAGARELA', f, repo_type='dataset', local_dir=RAW)
    if (k+1) % 5 == 0: print(f'  {k+1}/{N_SHARDS}  {time.time()-t0:.0f}s', flush=True)
print(f'{N_SHARDS} shards em {(time.time()-t0)/60:.1f} min')


### 5.1 Cuts + fbank

`prep_tagarela.py` filtra alucinação, gera o relatório de cobertura por show e grava features
em **lilcom_chunky** — 33 MB/h contra 115 MB/h do default numpy `[MEDIDO]`.


In [ ]:
!cd /content/jvscribe && python3 jvscribe/finetune/prep_tagarela.py \
    --parquet-dir /content/tagarela_raw/data \
    --out /content/data/tagarela \
    --num-jobs 4
!du -sh /content/data/tagarela/feats_train
!cat /content/data/tagarela/show_distribution.txt 2>/dev/null | head -5


### 5.2 Verificar o storage — não confie no parâmetro

O lhotse aceitou `storage_type` e gravou numpy assim mesmo numa execução real
(`wiki/medicoes/m10-t3-fbank-e-storage.md`). **Conferir custa um segundo; errar custa 410 GB**
na escala de 5.000 h.


In [ ]:
import gzip, json as _json
with gzip.open('/content/data/tagarela/tagarela_cuts_train.jsonl.gz','rt') as f:
    c = _json.loads(f.readline())
st = c['features']['storage_type']
print('storage_type =', st)
assert st == 'lilcom_chunky', f'ESPERADO lilcom_chunky, veio {st} -- 3,5x mais disco'


### 5.3 Nomear os manifests como o datamodule espera

O datamodule do CommonVoice carrega `{cv_manifest_dir}/cv-{language}_cuts_{split}.jsonl.gz`
(`asr_datamodule.py:409` e `:432`). Nossos cuts se chamam `tagarela_cuts_train` — e o próprio
`prep_tagarela.py` avisa, na docstring, que esse prefixo **não casa o glob default**.

Sem este passo o treino procura `data/en/fbank/cv-en_cuts_train.jsonl.gz` e morre.


In [ ]:
from lhotse import CutSet
CV, LANGID, N_DEV = '/content/data/tagarela', 'pt', 1000

cuts = CutSet.from_file(f'{CV}/tagarela_cuts_train.jsonl.gz')
ids  = list(cuts.ids)
assert len(ids) > N_DEV * 5, f'so {len(ids)} cuts -- dev de {N_DEV} come demais'

# Fatia CONTIGUA do fim. Os cut ids sao um contador na ordem em que os shards foram
# lidos, e um shard e um show, entao o fim tende a ser 1-2 shows.
# HONESTIDADE: isto NAO e disjuncao de show provada -- prep_tagarela.py conta o show
# para o relatorio mas NAO grava no cut, entao daqui nao da para verificar. Para medir
# CUSTO por epoca, que e o objetivo deste notebook, nao importa. Para comparar WER
# entre receitas, importa, e o campo precisa existir antes.
dev_ids = set(ids[-N_DEV:])
dev   = cuts.subset(cut_ids=list(dev_ids))
train = cuts.filter(lambda c: c.id not in dev_ids)

dev.to_file(f'{CV}/cv-{LANGID}_cuts_dev.jsonl.gz')
train.to_file(f'{CV}/cv-{LANGID}_cuts_train.jsonl.gz')

HORAS_TREINO = sum(c.duration for c in CutSet.from_file(f'{CV}/cv-{LANGID}_cuts_train.jsonl.gz'))/3600
HORAS_DEV    = sum(c.duration for c in dev)/3600
print(f'train {len(ids)-N_DEV} cuts / {HORAS_TREINO:.1f} h')
print(f'dev   {N_DEV} cuts / {HORAS_DEV:.1f} h')


## 6. Tokenizer BPE

⚠️ O `bpe.model` do M4 **se perdeu** e custou retrabalho. Aqui ele vai para o Drive junto dos
checkpoints, porque sem ele o checkpoint é inútil.


In [ ]:
LANG = '/content/data/lang_bpe_500'; os.makedirs(LANG, exist_ok=True)
import sentencepiece as spm

# user_defined_symbols NAO e opcional: train.py faz
#   params.blank_id = sp.piece_to_id("<blk>")            (train.py:1138)
# e, sem o simbolo, piece_to_id devolve o id do <unk>. Foi o que aconteceu na
# primeira tentativa -- o log trouxe "blank_id": 2, ou seja, blank COLIDINDO com unk.
# Nada falha; o modelo so treina errado. icefall define os dois simbolos primeiro,
# entao <blk>=0, <sos/eos>=1, <unk>=2 (local/train_bpe_model.py:85).
# pad_id foi removido: valia 0 e colidia com o <blk>.
spm.SentencePieceTrainer.train(
    input='/content/data/tagarela/transcript_words.txt',
    model_prefix=f'{LANG}/bpe', vocab_size=500, model_type='bpe',
    character_coverage=1.0,
    user_defined_symbols=['<blk>', '<sos/eos>'],
    unk_id=2, bos_id=-1, eos_id=-1)

_sp = spm.SentencePieceProcessor(model_file=f'{LANG}/bpe.model')
assert _sp.piece_to_id('<blk>') == 0, f"<blk> ficou em {_sp.piece_to_id('<blk>')}, deveria ser 0"
print('vocab', _sp.get_piece_size(), '| <blk>=0 ok')

!cp {LANG}/bpe.model {EXP}/   # sobrevive a sessao
print('bpe.model ->', EXP)


## 7. Comando de 66M — JÁ MEDIDO, pule

Esta seção produziu `0,85 h por época` em 2026-09-21 e está registrada em
`wiki/medicoes/m10-prototipo-custo-por-epoca.md`. Rodar de novo só repete um número que já
existe. **Vá para a seção 8.**

In [ ]:
UNIDADES_POR_HORA = {'L4': 4.8, 'A100': 13.0, 'T4': 2.0}   # [ESTIMATIVA] nao medido
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
un_h = next((v for k,v in UNIDADES_POR_HORA.items() if k in GPU), None)
print(f'GPU: {GPU} | unidades/h: {un_h}')

EPOCAS = 3        # suficiente para medir custo/epoca; NAO para convergir

# MAX_DUR e BASE_LR andam JUNTOS -- foi a combinacao dos dois que divergiu.
# A primeira tentativa usou 300 s com lr 0.03 e a loss subiu ate NaN no batch
# ~3300. O RESULTS.md do proprio recipe treina com batch efetivo de ~2200 s
# (world-size 2 x max-duration 1000, ou 4 x 550) e base-lr 0.045 -- razao
# lr/batch de 2.0e-5. A nossa era 1.0e-4: cinco vezes mais quente.
#   300 s foi escolhido por medo de OOM; o log mostrou pico de 9.090 MB numa
#   L4 de 24 GB, entao havia folga de 2,5x que nao estava sendo usada.
MAX_DUR = 700     # ~9 GB medidos em 300 s; sobe proporcional. Baixar se der OOM.
BASE_LR = 0.015   # 0.015/700 = 2.1e-5, a razao do recipe

# --cv-manifest-dir + --language: sem eles o datamodule procura
#   data/en/fbank/cv-en_cuts_train.jsonl.gz     (o default da receita CommonVoice)
# --enable-musan 0: com musan ligado ele carrega {manifest_dir}/musan_cuts.jsonl.gz
#   (asr_datamodule.py:231), que nao existe aqui.
# --use-transducer 1: o alvo do M10 e RNN-T streaming. A perda podada do transducer
#   e justamente a parte cara, entao medir so CTC produziria um custo que nao
#   transfere para a receita real.
cmd = (
    'cd /content/icefall/egs/commonvoice/ASR && python3 zipformer/train.py'
    f' --world-size 1 --num-epochs {EPOCAS} --start-epoch 1'
    f' --exp-dir {EXP} --bpe-model {LANG}/bpe.model'
    f' --cv-manifest-dir {CV} --language {LANGID}'
    ' --enable-musan 0'
    f' --max-duration {MAX_DUR} --use-fp16 0 --num-workers 2'
    ' --num-encoder-layers 2,2,3,4,3,2'
    ' --feedforward-dim 512,768,1024,1536,1024,768'
    ' --encoder-dim 192,256,384,512,384,256'
    ' --encoder-unmasked-dim 192,192,256,256,256,192'
    f' --causal 1 --use-transducer 1 --use-ctc 1 --base-lr {BASE_LR}'
)
print(cmd)


## 8. Uma época a ~200M — a escala por parâmetros

O run de 66,4M mede o custo de um modelo **do tamanho do que já está em produção**. A
faixa-alvo do M10 é **180–220M** (teto de 230M sob carga de softphone,
`wiki/medicoes/m10-rnf05-carga-concorrente.md`).

Extrapolar 66M → 200M por regra de três é `[ESTIMATIVA]` e provavelmente otimista: no
Zipformer, crescer em parâmetros alarga as dimensões, e a atenção é quadrática no
comprimento da sequência. **Uma época mede isso** — e é a maior incerteza que sobrou na
Fase 3.

A célula abaixo **conta os parâmetros antes de treinar**, em vez de escolher dimensões no
escuro. Constrói cada candidato na CPU, mede, e só então monta o comando.


In [ ]:
# --- escolher as dimensoes MEDINDO, nao adivinhando -------------------------
# get_parser() ja chama add_model_arguments(); o datamodule so entra no main(),
# entao parse_args() aqui aceita apenas as flags de modelo. Nenhum argumento e
# required=True, e blank_id/vocab_size sao preenchidos no main a partir do
# bpe.model -- por isso os dois entram a mao.
# Esta celula pode rodar sozinha DEPOIS das de setup, mas depende do que elas
# deixaram na memoria. Falhar aqui custa segundos; falhar na celula 21 custa a
# hora de treino inteira, porque o custo so e impresso no fim.
_faltando = [n for n in ('EXP','LANG','CV','LANGID','HORAS_TREINO') if n not in dir()]
if _faltando:
    raise NameError(f'variaveis ausentes: {_faltando}. A sessao reiniciou? '
                    'Rode as celulas 2, 6, 8, 16 e 18 antes desta.')

# un_h nasce na celula 20, que esta celula substitui. Sem ele a 21 quebra com
# NameError na linha que imprime o custo -- depois do treino inteiro.
UNIDADES_POR_HORA = {'L4': 4.8, 'A100': 13.0, 'T4': 2.0}   # [ESTIMATIVA] nao medido
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
un_h = next((v for k,v in UNIDADES_POR_HORA.items() if k in GPU), None)
print(f'GPU: {GPU} | unidades/h: {un_h} | corpus: {HORAS_TREINO:.1f} h\n')

sys.path.insert(0, '/content/icefall/egs/commonvoice/ASR/zipformer')
import train as T

CANDIDATOS = {
 'medium (o run de 66M)': dict(
    layers='2,2,3,4,3,2', ffw='512,768,1024,1536,1024,768',
    dim='192,256,384,512,384,256', unmask='192,192,256,256,256,192'),
 'large (icefall)': dict(
    layers='2,2,4,5,4,2', ffw='512,768,1536,2048,1536,768',
    dim='192,256,512,768,512,256', unmask='192,192,256,320,256,192'),
 'large + camadas': dict(
    layers='2,3,4,6,4,3', ffw='512,768,1536,2048,1536,768',
    dim='192,256,512,768,512,256', unmask='192,192,256,320,256,192'),
 'large + largura': dict(
    layers='2,2,4,5,4,2', ffw='768,1024,1536,2560,1536,1024',
    dim='256,384,512,896,512,384', unmask='192,192,256,384,256,192'),
 'xlarge': dict(
    layers='2,3,4,6,4,3', ffw='768,1024,2048,2560,2048,1024',
    dim='256,384,640,896,640,384', unmask='192,192,320,384,320,192'),
}
ALVO_M = 200

def flags_de(c):
    return ['--num-encoder-layers', c['layers'], '--feedforward-dim', c['ffw'],
            '--encoder-dim', c['dim'], '--encoder-unmasked-dim', c['unmask'],
            '--causal', '1', '--use-transducer', '1', '--use-ctc', '1']

def conta(c):
    p = T.get_params(); p.update(vars(T.get_parser().parse_args(flags_de(c))))
    p.blank_id, p.vocab_size = 0, 500          # o BPE desta sessao
    return sum(x.numel() for x in T.get_model(p).parameters()) / 1e6

medidos = {}
for nome, c in CANDIDATOS.items():
    medidos[nome] = conta(c)
    print(f'  {medidos[nome]:7.1f}M  {nome}')

nome = min(medidos, key=lambda k: abs(medidos[k] - ALVO_M))
CFG, P_M = CANDIDATOS[nome], medidos[nome]
print(f'\nescolhido: {nome} -- {P_M:.1f}M (alvo {ALVO_M}M)')

# --- memoria e a razao lr/batch --------------------------------------------
# UM ponto medido: 9.090 MB com 66,4M a max-duration 300 (log do run que divergiu).
# Modelo: memoria = fixo(P) + ativacoes(P, T).
#   fixo        = P x 4 bytes x 4 tensores (peso, grad, 2 estados do ScaledAdam)
#   ativacoes  ~ largura x T, e largura ~ sqrt(P), entao ~ sqrt(P) x T
# [ESTIMATIVA] de um unico ponto. Se der OOM, corte MAX_DUR pela metade -- o
# BASE_LR desce junto, sozinho, porque o que importa e a RAZAO.
FIXO_MB  = lambda p: p*1e6*4*4/1e6
ATIV_300 = 9090 - FIXO_MB(66.4)                      # 8.030 MB a 66,4M/300 s
ORCAMENTO_MB = 20000                                 # 23.034 na L4, ~3 GB de folga
MAX_DUR = int(300 * (ORCAMENTO_MB - FIXO_MB(P_M)) / ATIV_300 / (P_M/66.4)**0.5 / 10) * 10

# A razao lr/batch e o que quebrou a primeira tentativa (1,0e-4 contra 2,0e-5 do
# recipe). Ela fica fixa aqui; o lr e consequencia do batch, nunca escolhido solto.
RAZAO  = 2.1e-5
BASE_LR = round(RAZAO * MAX_DUR, 5)
EPOCAS  = 1     # o objetivo e o custo, nao o modelo

print(f'fixo estimado {FIXO_MB(P_M)/1000:.1f} GB | MAX_DUR {MAX_DUR} s [ESTIMATIVA]'
      f' | BASE_LR {BASE_LR} (razao {RAZAO:.1e})')

EXP200 = EXP.replace('/exp', '/exp-200m')   # nao mistura com os checkpoints de 66M
os.makedirs(EXP200, exist_ok=True)
cmd = (
    'cd /content/icefall/egs/commonvoice/ASR && python3 zipformer/train.py'
    f' --world-size 1 --num-epochs {EPOCAS} --start-epoch 1'
    f' --exp-dir {EXP200} --bpe-model {LANG}/bpe.model'
    f' --cv-manifest-dir {CV} --language {LANGID}'
    ' --enable-musan 0'
    f' --max-duration {MAX_DUR} --use-fp16 0 --num-workers 2'
    f" --num-encoder-layers {CFG['layers']}"
    f" --feedforward-dim {CFG['ffw']}"
    f" --encoder-dim {CFG['dim']}"
    f" --encoder-unmasked-dim {CFG['unmask']}"
    f' --causal 1 --use-transducer 1 --use-ctc 1 --base-lr {BASE_LR}'
)
EXP = EXP200          # a celula 21 le EXP para contar checkpoints
print('\n' + cmd)


## 9. Rodar e medir

Roda o `cmd` que a seção acima montou, com vigia de divergência, e imprime o custo por
época. **Uma época a ~200M leva algumas horas** — a barra de progresso é o log filtrado.


In [ ]:
# `!{cmd}` engole o exit code: na primeira tentativa o treino morreu em 40 s e esta
# celula imprimiu "$0.00 por epoca" como se fosse medicao. Um numero sobre um run
# que falhou nao mede nada -- entao o exit code manda.
#
# E o log vai para arquivo: na segunda tentativa o treino rodou 77 min e a excecao
# nao dizia por que. 77 min de saida nao cabem no output da celula.
#
# A terceira tentativa divergiu: a loss SUBIU por 3.300 batches ate virar NaN, e
# so entao o icefall abortou -- 75 min gastos depois que o problema ja era visivel.
# Por isso ha um vigia: a tot_loss de cada log e comparada com a menor ja vista, e
# o treino e morto se subir 25% e ficar assim. Custo/epoca ja estara medido.
import re, signal

LOG = '/content/train.log'
print(f'log -> {LOG}  (tail -f em outra celula para acompanhar)')

RE_TOT = re.compile(r'tot_loss\[loss=([\d.]+)')
melhor, ruins, MAX_RUINS, FATOR = float('inf'), 0, 4, 1.25

t0 = time.time()
with open(LOG, 'w') as fh:
    proc = subprocess.Popen(['bash', '-lc', cmd], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1,
                            start_new_session=True)
    divergiu = False
    for linha in proc.stdout:
        fh.write(linha); fh.flush()
        if any(k in linha for k in ('Epoch', 'Error', 'error', 'Traceback',
                                    'CUDA', 'Saving', 'validation',
                                    'Maximum memory', 'model parameters')):
            print(linha, end='')
        m = RE_TOT.search(linha)
        if m:
            v = float(m.group(1))
            if v != v or v > melhor * FATOR:      # NaN ou subiu demais
                ruins += 1
                if ruins >= MAX_RUINS:
                    divergiu = True
                    print(f'\n!! DIVERGIU: tot_loss {v:.3f} contra minimo {melhor:.3f} '
                          f'em {MAX_RUINS} logs seguidos. Matando o treino.')
                    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
                    break
            else:
                ruins = 0
                melhor = min(melhor, v)
    proc.wait()
el = time.time() - t0

# O custo por epoca e uma medida de THROUGHPUT, nao de qualidade: o forward e o
# backward custam o mesmo numa epoca que converge e numa que diverge. Entao o
# numero vale mesmo aqui -- desde que o run tenha durado pelo menos uma epoca.
ckpts = sorted(f for f in os.listdir(EXP) if f.startswith('epoch-') and f.endswith('.pt'))
for f in ckpts:
    print(f'  {f}  {os.path.getsize(os.path.join(EXP, f))/1e6:.0f} MB')

if not ckpts:
    cauda = open(LOG).read().splitlines()[-40:]
    print('\n'.join(cauda))
    raise RuntimeError(f'nenhuma epoca completou em {el/60:.1f} min -- sem custo a medir. '
                       f'Log em {LOG}.')

h_por_epoca = el/3600/len(ckpts)
print(f'\n=== CUSTO MEDIDO ===   ({HORAS_TREINO:.0f} h de corpus, {len(ckpts)} epoca(s))')
print(f'{h_por_epoca:.2f} h por epoca')
if un_h:
    print(f'{h_por_epoca*un_h:.1f} unidades = ${h_por_epoca*un_h/10:.2f} por epoca')
    for horas_corpus in (1500, 5000):
        fator = horas_corpus/HORAS_TREINO
        print(f'  extrapolado p/ {horas_corpus} h x 10 epocas: '
              f'${h_por_epoca*un_h/10*fator*10:.0f}  [ESTIMATIVA linear]')

if divergiu or proc.returncode != 0:
    print(f'\nO CUSTO ACIMA VALE. O MODELO NAO: exit {proc.returncode}, divergiu={divergiu}.')


### 9.1 Se falhou: onde o treino parou

O icefall grava seu próprio log em `{exp-dir}/log/`, e o exp-dir está no **Drive** — então
ele sobrevive à sessão, mesmo quando o output da célula se perde.


In [ ]:
# Forense do run que falhou. O exp-dir esta no Drive, entao SOBREVIVE a sessao:
# o icefall grava log/log-train-* la, e e ele que diz onde o treino parou.
import glob, os

print('=== o que sobrou no exp-dir (Drive) ===')
for f in sorted(glob.glob(f'{EXP}/**/*', recursive=True)):
    if os.path.isfile(f):
        print(f'  {os.path.getsize(f)/1e6:9.1f} MB  {f.replace(EXP+"/", "")}')

logs = sorted(glob.glob(f'{EXP}/log/log-train-*'), key=os.path.getmtime)
if not logs:
    print('\nsem log/log-train-* -- o treino morreu antes de abrir o log')
else:
    print(f'\n=== ultimas 40 linhas de {os.path.basename(logs[-1])} ===')
    print('\n'.join(open(logs[-1], errors="replace").read().splitlines()[-40:]))

print('\n=== disco da sessao ===')
!df -h /content | tail -1
!du -sh /content/data/tagarela/* 2>/dev/null | sort -h | tail -5


## 10. O que fazer com o número

1. Registrar em `wiki/medicoes/` com a GPU, o corpus e as épocas ao lado — sem isso o número
   não transfere.
2. Reprecificar a Fase 3 de `docs/plans/m10-treino-vastai.md`, substituindo a proporção
   herdada de M5 pela medição.
3. **A extrapolação linear é otimista**: batch maior aproveita melhor a GPU, e corpus maior
   muda o gargalo de compute para I/O. Tratar como piso, não como estimativa.

### Se der OOM

Baixar `--max-duration` antes de qualquer outra coisa. O registro de M5 tem OOM em pre-scan
resolvido com `eager` + `workers=2`.

### Se a loss platôar em blank e o WER for 100%

É a armadilha do LR de cabeça fresca. Conferir que `--base-lr` está em 0.03 e não em 0.0001.
